# Understand an indoor scene

A robot mapping an office, or a facilities tool auditing a floor, does not want a tensor back. It wants an *inventory*: what is in this room, how much floor each thing takes, where every object sits. No single checkpoint produces that. You get there by running a semantic segmenter and a 3D detector over the same room and reconciling what the two say.

This notebook does exactly that on one real reconstructed room, committed with the docs, and ends with a list you could hand to a planner:

- the room as it arrives, and the six numbers per point a checkpoint actually reads
- a label for every original point, scored against the room's own annotation
- oriented boxes for the objects, and a check of each box against the labels inside it
- the inventory: floor area per class, objects per class, sizes
- what to do when the model's label set is not the one you need

It assumes you can already run a segmenter through an `Inferer`. If that is new, read [Segment a scene](02-segmentation-inference.md) first.

!!! note

    Everything except the last section runs from the committed room. It needs the two pretrained checkpoints in the local model cache and the `spconv` extra for the sparse-convolution backbone. The last section additionally needs the full ScanNet download and is marked as such.

In [ ]:
# On Colab: !pip install "torch-pointcloud[pyg-lib]"
import numpy as np
import torch
from plyfile import PlyData

import torch_pointcloud as tp
import torch_pointcloud.transforms as T

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch-pointcloud", tp.__version__, "| device:", device)

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection

from torch_pointcloud.utils.box3d import box_corners

VIEW = (39, -160)  # elevation and azimuth: in through the room's open corner, which a default camera misses
BOX_EDGES = ((0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7))


def show_cloud(pos, color=None, *, ax=None, title=None, size=0.8, view=VIEW, cmap="tab20", vmin=None, vmax=None):
    """Scatter a room. `pos` is (N, 3); `color` is per-point RGB on 0-1, a label vector, or None.

    Pass `vmin` and `vmax` whenever two panels show label vectors that must agree: without them
    matplotlib normalizes each panel against its own range and the same class comes out two colors.
    """
    if ax is None:
        ax = plt.figure(figsize=(6, 5)).add_subplot(projection="3d")
    p = pos.cpu().numpy()
    c = color.cpu().numpy() if torch.is_tensor(color) else color
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=c, s=size, cmap=cmap, vmin=vmin, vmax=vmax,
               depthshade=False, linewidths=0)
    ax.view_init(elev=view[0], azim=view[1])
    ax.set_box_aspect(p.max(0) - p.min(0))
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=10)
    return ax


def show_boxes(ax, boxes, colors):
    """Draw (K, 7) oriented boxes on a 3D axes as one wireframe per row, heading included."""
    for corners, color in zip(box_corners(boxes).cpu().numpy(), colors):
        segments = [(corners[a], corners[b]) for a, b in BOX_EDGES]
        ax.add_collection3d(Line3DCollection(segments, colors=color, linewidths=1.2))

## The room as it arrives

`docs/assets/data/sample_scene_labeled.ply` is one ScanNet room: a triangle mesh reconstructed from an RGB-D scan, kept as its vertices, axis-aligned and recentered on the origin. Every vertex carries a color, a NYU40 semantic id and an instance id.

In [ ]:
import urllib.request
from pathlib import Path

path = Path("../assets/data/sample_scene_labeled.ply")  # in a docs checkout
if not path.exists():
    path = Path("sample_scene_labeled.ply")
    url = "https://github.com/arthurdjn/pytorch-pointcloud/raw/main/docs/assets/data/sample_scene_labeled.ply"
    if not path.exists():
        urllib.request.urlretrieve(url, path)

vertex = PlyData.read(path)["vertex"]
room = {
    "pos": torch.from_numpy(np.stack([vertex["x"], vertex["y"], vertex["z"]], axis=1).astype(np.float32)),
    "color": torch.from_numpy(np.stack([vertex["red"], vertex["green"], vertex["blue"]], axis=1).astype(np.float32)),
    "segment": torch.from_numpy(np.asarray(vertex["segment"]).astype(np.int64)),
    "instance": torch.from_numpy(np.asarray(vertex["instance"]).astype(np.int64)),
}

extent = room["pos"].max(0).values - room["pos"].min(0).values
print("points:", len(room["pos"]))
print("extent (m):", [round(float(value), 2) for value in extent])
print("color range:", float(room["color"].min()), "-", float(room["color"].max()))
print("annotated objects:", len(set(room["instance"].tolist()) - {-1}))

That is 127 410 points over $6.26 \times 8.46 \times 2.62$ m, with 33 annotated objects: one meeting room, dense enough that a 2 cm voxel grid keeps almost every point.

Two things are missing before a checkpoint can read it.

**Normals.** The ScanNet20 segmentation checkpoints take six features per point, $[r, g, b, n_x, n_y, n_z]$ on 0-1 and unit length. The committed file carries no normals, so estimate them from local geometry with `EstimateNormals`: each normal is the least-variance direction of a point's $k$ nearest neighbors.

**A shared label set.** The file's `segment` holds raw NYU40 ids. The checkpoint was trained on the 20-class ScanNet benchmark, whose ids agree with NYU40 up to `counter` and then diverge: a chair is $5$ in both, but a refrigerator is $24$ in NYU40 and $15$ in the benchmark. Feeding the raw ids straight in would therefore look almost right while silently discarding this room's refrigerator, sink and otherfurniture points, 6 890 of them. `ScanNet20` publishes the mapping as `SCANNET20_LABELS`, so one `Relabel` bridges the two. The section [When the label set is not yours](#when-the-label-set-is-not-yours) comes back to this.

In [ ]:
from torch_pointcloud.datasets.scannet import SCANNET20_LABELS

room = T.EstimateNormals(keys="pos", k=16)(room)
print("normal:", tuple(room["normal"].shape), "| unit length:", float(room["normal"].norm(dim=1).mean()))

print("NYU40 ids in the room:", sorted(set(room["segment"].tolist())))
room = T.Relabel(keys="segment", labels=SCANNET20_LABELS)(room)
print("after Relabel:", sorted(set(room["segment"].tolist())))

In [ ]:
fig = plt.figure(figsize=(12, 5))
show_cloud(room["pos"], room["color"] / 255, ax=fig.add_subplot(121, projection="3d"),
           title="color, as the room was reconstructed")
show_cloud(room["pos"], room["normal"].abs(), ax=fig.add_subplot(122, projection="3d"),
           title="normal, estimated from 16 neighbors");

![The committed room drawn twice, once in its reconstruction color and once by estimated surface normal](../assets/tutorials/indoor_room.png)

Those two panels are the six input channels. Look at how cleanly the estimated normals separate the floor from the two wall directions: that is the geometry cue the color alone does not give.

Both panels look in through the room's open corner. A camera pointed at the default angle looks at this room through a wall.

## A label for every point

`create_model(..., return_info=True)` hands back the checkpoint *and* the exact preprocessing it was trained with, plus the class names it predicts.

In [ ]:
model, info = tp.create_model(
    "spunet-v1m1.scannet20.pointcept",
    task="segmentation",
    pretrained=True,
    return_info=True,
)
model = model.eval().to(device)
classes = list(info["weights"]["classes"])
print(len(classes), "classes:", ", ".join(classes))

The transform voxelizes the room at 2 cm, builds `x` from color and normal, and writes an `inverse` map from every original point to the voxel it landed in. It also copies the labels to `origin_segment` *before* voxelizing, which is what makes a full-resolution score possible.

In [ ]:
from torch_pointcloud.utils.data import collate

batch = collate([info["transform"]({key: value.clone() for key, value in room.items()})])
print({key: tuple(value.shape) for key, value in batch.items() if torch.is_tensor(value)})

127 410 points become 114 118 voxels. An [inferer](../inferers/overview.md) takes the batch and a `predictor` callable and returns one row per *input* point, so the predictor is where the voxel logits are broadcast back through `inverse`. The room fits in one forward pass, so `SimpleInferer` is enough; a scene that does not fit would swap in `SlidingWindowInferer` with no other change.

In [ ]:
from torch_pointcloud.inferers import SimpleInferer


def predictor(data):
    """Voxel logits, broadcast back to one row per original point through the `inverse` map."""
    logits = model(data["x"].to(device), data["pos_grid"].to(device), data["batch"].to(device))
    return logits[data["inverse"].to(device)]


with torch.no_grad():
    probabilities = SimpleInferer(softmax=True)(batch, predictor=predictor).cpu()

labels = probabilities.argmax(dim=-1)
print("probabilities:", tuple(probabilities.shape))
print("one row per input point:", len(probabilities) == len(room["pos"]))

Alignment confirmed, `labels[i]` is the class of `room["pos"][i]`. Score it against `origin_segment`, which is the room's own annotation at full resolution. 18 576 of the points carry no label at all (NYU40 id $0$, plus the classes outside the benchmark's 20), and `ignore_index=-1` keeps them out of both metrics.

In [ ]:
from torch_pointcloud.utils.metrics import confusion_matrix

target = batch["origin_segment"]
annotated = target >= 0
matrix = confusion_matrix(labels, target, model.num_classes, ignore_index=-1)
union = matrix.sum(0) + matrix.sum(1) - matrix.diag()
iou = matrix.diag() / union.clamp_min(1)
accuracy = (labels[annotated] == target[annotated]).float().mean()
miou = iou[union > 0].mean()

print(f"annotated points: {int(annotated.sum())} of {len(target)}")
print(f"accuracy: {accuracy:.3f}")
print(f"mIoU over the {int((union > 0).sum())} classes present: {miou:.3f}")
for index in torch.nonzero(union > 0).flatten().tolist():
    print(f"  {classes[index]:14s} IoU {float(iou[index]):.3f}")

In [ ]:
fig = plt.figure(figsize=(12, 5))
shared = {"vmin": -1, "vmax": len(classes) - 1}  # one color scale over both panels, ignore index included
show_cloud(room["pos"], labels, ax=fig.add_subplot(121, projection="3d"), **shared,
           title=f"predicted: {accuracy:.0%} correct, mIoU {miou:.2f}")
show_cloud(room["pos"], target, ax=fig.add_subplot(122, projection="3d"), **shared,
           title="ground truth, ScanNet20 ids");

![The room colored by predicted ScanNet20 labels next to the same room colored by its own annotation](../assets/tutorials/indoor_semantics.png)

Both panels share one legend, so a class keeps its color across them and the two answers compare hue for hue. The gray points on the right are the ones the room carries no label for.

80.1% of the annotated points are right, at a mIoU of 0.416 over the 15 classes involved. Read those two numbers together, because they disagree on purpose: floor scores 0.967 and chair 0.864, while refrigerator, bookshelf, curtain and picture all score 0.000. A per-scene mIoU divides by whichever classes appear, so one missed class costs it far more than it costs accuracy.

!!! note

    Estimated normals are an approximation of the mesh normals the checkpoint trained on, and they cost real accuracy. On the same scene loaded from the full ScanNet download, this checkpoint scores 0.875 accuracy with the mesh normals and 0.689 when the normals are re-estimated the way they are here. Keep the reconstruction's normals when you have them.

In [ ]:
verdict = torch.where(annotated, (labels == target).long(), torch.full_like(target, -1))

show_cloud(room["pos"], verdict, cmap="RdYlGn", vmin=-1, vmax=1,
           title=f"{float((verdict == 0).sum() / annotated.sum()):.0%} of the labeled points differ");

The errors are not spread evenly. They concentrate on the walls and the surfaces hung on them, and stay off the floor and the chairs, which is exactly what the per-class IoU says: wall 0.524 against floor 0.967 and chair 0.864. The unannotated points keep the lowest color of the scale, which is how the 18 576 of them stay visible without being counted.

## Objects, not just points

Per-point labels tell you *chair-ness* is present. They do not tell you there are ten chairs. For that, run a detector over the same room. `votenet.scannet.fair` predicts the standard 18-class ScanNet detection set, and its own transform subsamples the room to 40 000 points and appends a height feature.

In [ ]:
detector, detector_info = tp.create_model(
    "votenet.scannet.fair",
    task="detection",
    pretrained=True,
    return_info=True,
)
detector = detector.eval().to(device)
detector_classes = list(detector_info["weights"]["classes"])
print(len(detector_classes), "classes:", ", ".join(detector_classes))

torch.manual_seed(0)
detection_batch = collate([detector_info["transform"]({"pos": room["pos"].clone()})])
with torch.no_grad():
    out = detector(
        detection_batch["x"].to(device),
        detection_batch["pos"].to(device),
        detection_batch["batch"].to(device),
    )
detections = detector.decode(out)
print({key: tuple(value.shape) for key, value in detections.items()})

`decode` returns the raw proposal set, 256 boxes, one per vote cluster, most of them duplicates or empty air. Two filters turn it into an object list. `nms3d` drops boxes that overlap a better-scoring box of the same class, and `count_points_in_boxes` counts how many room points each surviving box actually contains, which removes proposals floating in space that a score threshold alone would keep.

In [ ]:
from torch_pointcloud.utils.box3d import count_points_in_boxes, nms3d

boxes = detections["boxes"].cpu()
scores = detections["scores"].cpu()
box_labels = detections["labels"].cpu()

keep = nms3d(boxes, scores, 0.25, labels=box_labels)
boxes, scores, box_labels = boxes[keep], scores[keep], box_labels[keep]
print("after NMS:", len(boxes))

counts = count_points_in_boxes(room["pos"], boxes)
keep = ((scores > 0.9) & (counts >= 50)).nonzero().squeeze(-1)
keep = keep[torch.argsort(scores[keep], descending=True)]
boxes, scores, box_labels, counts = boxes[keep], scores[keep], box_labels[keep], counts[keep]
print("confident boxes holding points:", len(boxes))

for box, score, label, count in zip(boxes, scores, box_labels, counts):
    size = " x ".join(f"{value:.2f}" for value in box[3:6].tolist())
    print(f"  {detector_classes[label]:9s} score {score:.2f}  {count:6d} points  {size} m")

In [ ]:
CLASS_COLOR = [f"C{index % 10}" for index in range(len(detector_classes))]

ax = show_cloud(room["pos"], room["color"] / 255, size=1.0,
                title=f"{len(boxes)} boxes above score 0.9, holding at least 50 points")
show_boxes(ax, boxes, [CLASS_COLOR[int(label)] for label in box_labels]);

![The room under eighteen wireframe boxes, each colored by its detected class](../assets/tutorials/indoor_detection.png)

`box_corners` turns the $(K, 7)$ rows into the eight corners of each box, heading included, which is all a wireframe needs. Look at where the boxes land: ten of them sit on individual chairs around the long table, and the two wall cabinets, the door and the window each get one.

The right panel is the same boxes from straight above, which is the view that answers *where in the room*: the ten chairs line up along both sides of the table, and the counter and cabinets run along the far wall.

## Do the two answers agree?

256 proposals become 104 after NMS and 18 after the score and point floors. Each box is $[c_x, c_y, c_z, d_x, d_y, d_z, \theta]$ with full extents and a heading about $+z$.

Now cross-check them. `points_in_oriented_box` gives the mask of room points inside one box, so for every box you can ask which class most of its points were *labeled*. Wall and floor are dropped first: they are the bulk of what any box contains and are never the object it is about.

In [ ]:
from torch_pointcloud.transforms import functional as F

shell = torch.tensor([classes.index(name) for name in ("wall", "floor")])
majority, agrees = [], []
for box, label in zip(boxes, box_labels):
    inside = F.points_in_oriented_box(room["pos"], torch.cat([box[:3], box[3:6] / 2, box[6:7]]))
    objects = labels[inside][~torch.isin(labels[inside], shell)]
    top = classes[int(objects.bincount().argmax())] if len(objects) else "none"
    majority.append(top)
    agrees.append(top == detector_classes[label])

print(f"{sum(agrees)} of {len(boxes)} boxes match the labels inside them")
for label, top, ok in zip(box_labels.tolist(), majority, agrees):
    print(f"  box {detector_classes[label]:9s} -> most points labeled {top:9s} {'match' if ok else 'differs'}")

13 of the 18 boxes agree with the labels inside them, and the five that differ are worth reading one by one rather than dismissing:

- the box on the long **table** is 3.56 m long and swallows the chairs pulled up to it, so *chair* wins the vote inside it. A box that contains its neighbors is a framing problem, not an error.
- both **cabinet** boxes come back as *counter*, and the **sink** box comes back as *counter* too. The sink is set into the counter, and cabinet against counter is a taxonomy boundary the two checkpoints draw in different places.
- the **window** box comes back as *cabinet*. That wall is where the segmenter is weakest: window scores 0.519 IoU and cabinet 0.117.

That is the value of the cross-check: it separates a detector that framed an object loosely from a segmenter that got a surface wrong, and neither model can tell you which on its own.

## An inventory of the room

Both outputs now become numbers a downstream tool consumes. Per-point labels give coverage: bin the points onto a 5 cm ground-plane grid and count the occupied cells, and the result is the floor area each class covers, in square meters. Boxes give the object count and each object's size.

In [ ]:
cells = torch.floor(room["pos"][:, :2] / 0.05).long()
inventory = []
for index in labels.unique().tolist():
    if classes[index] in ("wall", "floor"):
        continue
    footprint = len(torch.unique(cells[labels == index], dim=0)) * 0.05**2
    inventory.append((classes[index], int((labels == index).sum()), footprint))

inventory.sort(key=lambda row: -row[2])
for name, points, footprint in inventory[:6]:
    print(f"  {name:14s} {points:6d} points  {footprint:5.2f} m2")

found = torch.bincount(box_labels, minlength=len(detector_classes))
print("objects:", {detector_classes[index]: int(n) for index, n in enumerate(found) if n})

top = inventory[:8][::-1]
ax = plt.figure(figsize=(7, 3.5)).add_subplot()
ax.barh([name for name, _, _ in top], [area for _, _, area in top], color="tab:orange", height=0.62)
ax.set_xlabel("floor covered (m$^2$)")
ax.spines[["top", "right"]].set_visible(False);

![A horizontal bar chart of the floor area each predicted class covers, table highest at 4.41 square meters](../assets/tutorials/indoor_inventory.png)

Read the bars against the box counts: coverage and object count are two independent measurements of the same room.

| class | points | floor covered | boxes |
| --- | --- | --- | --- |
| table | 8 909 | 4.41 $m^2$ | 2 |
| chair | 19 400 | 3.26 $m^2$ | 10 |
| counter | 2 686 | 2.05 $m^2$ | 1 |
| cabinet | 6 912 | 1.11 $m^2$ | 2 |
| window | 3 783 | 0.76 $m^2$ | 1 |
| door | 4 023 | 0.40 $m^2$ | 1 |

`sink` sits ninth on the coverage list, 0.08 $m^2$ over 177 points, and still earns a box of its own: a small object is still an object.

Read as an inventory: *one meeting room, 6.3 by 8.5 m, ten chairs around a 3.6 m table, two wall cabinets, a counter with a sink, one door, one window.* The two halves are independent measurements of the same room, which is why quoting both is worth more than either: table covers the most floor at 4.41 $m^2$ but is only two objects, while chair covers 3.26 $m^2$ spread over ten.

## When the label set is not yours

Neither checkpoint speaks your schema, and they do not fully speak each other's either.

In [ ]:
print("detector only: ", sorted(set(detector_classes) - set(classes)))
print("segmenter only:", sorted(set(classes) - set(detector_classes)))

The detector has `garbagebin`, which the segmenter folds into `otherfurniture`. The segmenter has `wall` and `floor`, which a detector has no reason to box. Everything else lines up by name, which is what made the cross-check above possible.

Mapping onto a schema of your own is the same `Relabel` used on the input, pointed the other way: a dict from the checkpoint's class index to yours, with `default=-1` for everything you do not want.

In [ ]:
ROBOT_SCHEMA = {
    "chair": "seat",
    "sofa": "seat",
    "table": "work surface",
    "desk": "work surface",
    "counter": "work surface",
    "cabinet": "storage",
    "bookshelf": "storage",
    "refrigerator": "storage",
    "door": "opening",
    "window": "opening",
}

downstream = sorted(set(ROBOT_SCHEMA.values()))
to_downstream = {classes.index(name): downstream.index(value) for name, value in ROBOT_SCHEMA.items()}
coarse = T.Relabel(keys="segment", labels=to_downstream, default=-1)({"segment": labels})["segment"]

for index, name in enumerate(downstream):
    print(f"  {name:14s} {int((coarse == index).sum()):6d} points")
print(f"  {'dropped':14s} {int((coarse < 0).sum()):6d} points")

!!! warning "Remapping cannot add a class"

    Merging classes is safe: `chair` and `sofa` both become `seat`, and nothing is lost that the checkpoint knew. Splitting one is not, and neither is inventing one. This checkpoint has never seen a whiteboard, a projector or a potted plant, so no mapping will make them appear: their points land in `otherfurniture`, or on whichever of the 20 classes looks nearest. If your schema needs a class the training set did not have, that is a fine-tuning problem, not a mapping problem. [Train a model](05-training.md) covers it.

## The same room, from the full dataset

Everything above ran from one committed file. Point the same code at the `ScanNet20` dataset and it runs unchanged over all 312 validation rooms, with the reconstruction's own mesh normals instead of estimated ones. This needs the ScanNet download, so it is here for reference.

In [ ]:
from torch_pointcloud.datasets import ScanNet20

dataset = ScanNet20(root="data", split="val", transform=info["transform"])
batch = collate([dataset[0]])  # dataset[0] is scene0011_00, the room used above

with torch.no_grad():
    probabilities = SimpleInferer(softmax=True)(batch, predictor=predictor).cpu()

target = batch["origin_segment"]
annotated = target >= 0
print(f"accuracy: {(probabilities.argmax(-1)[annotated] == target[annotated]).float().mean():.3f}")

The dataset version of this room has 237 360 points against the committed file's 127 410, which is voxel-downsampled for size, and it carries mesh normals. Measured on it: **0.875** accuracy and **0.544** mIoU with the mesh normals, against **0.689** and **0.381** when the normals are re-estimated with `EstimateNormals(k=16)`. The pipeline is the same; the input features are not.

## Next steps

- [Segment a scene](02-segmentation-inference.md) for the inferer contract and the tiling strategies a room too large for one pass needs.
- [Segment a LiDAR sweep](07-large-scale-lidar.md) and [Detect objects while driving](08-driving-detection.md) for the same two tasks outdoors, where the scale changes what works.
- [Train a model](05-training.md) to fine-tune a checkpoint onto a class set of your own.
- [Search a scene by feature](06-feature-search.md) to find objects without a class list at all.